# Import Packages

In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
import itertools

import logging
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('pytorch_lightning').setLevel(logging.ERROR)

import xgboost as xgb
from xgboost import plot_importance

# Modeling Daily Rat Sightings in Manhattan

## Importing the Data

In [3]:
# set up the time series split
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2026-02-28.
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to MANHATTAN

rs = rs[rs['borough']=='MANHATTAN']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

rs

,created_date,count
2,2020-01-01,4
7,2020-01-02,7
12,2020-01-03,16
17,2020-01-04,10
21,2020-01-05,5
...,...,...
10611,2026-02-24,7
10616,2026-02-25,8
10620,2026-02-26,9
10625,2026-02-27,17


In [4]:
## There are 2251 days from 2020-01-01 to 2026-02-28 inclusive. So if this number is <2251, we must make sure to add in 0's.

len(rs)

2250

In [5]:
## We find the missing row and add it in.
## There's probably a better way to find in the missing dates and update it. This is a sort of hacky solution.

date_range = pd.date_range(start=rs['created_date'].min(), end=rs['created_date'].max(), freq='D')
complete_dates_df = pd.DataFrame(date_range, columns=['created_date'])
rs = pd.merge(complete_dates_df, rs, on='created_date', how='left')
rs['count'] = rs['count'].fillna(0).astype(int)
rs = rs.sort_values(by='created_date').reset_index(drop=True)

rs

,created_date,count
0,2020-01-01,4
1,2020-01-02,7
2,2020-01-03,16
3,2020-01-04,10
4,2020-01-05,5
...,...,...
2246,2026-02-24,7
2247,2026-02-25,8
2248,2026-02-26,9
2249,2026-02-27,17


## Baseline Seasonal Average Model

In [6]:
years_back_use = 4
day_window_use = 4

In [7]:
def seasonal_average_forecast(data, target_dates, years_back=years_back_use, day_window=day_window_use):
    df = data.copy()
    # ensure datetime type
    df["created_date"] = pd.to_datetime(df["created_date"])
    df["doy"] = df["created_date"].dt.dayofyear
    df["year"] = df["created_date"].dt.year

    forecasts = []
    for target_date in target_dates:
        target_doy = target_date.dayofyear
        target_year = target_date.year
        mask = ((df["year"] >= target_year - years_back) & (df["year"] < target_year) & (np.abs(df["doy"] - target_doy) <= day_window))
        forecasts.append(df.loc[mask, "count"].mean())
    return pd.Series(forecasts, index=target_dates)

In [8]:
results = []

rs["created_date"] = pd.to_datetime(rs["created_date"])

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    
    train = rs.iloc[train_index].copy()
    test = rs.iloc[test_index].copy()
    
    # Target dates = the dates we want to forecast. There are 14 days.
    target_dates = test["created_date"]
    
    # Seasonal forecast using only the training data (we will go back 5 years and take the average and use a day_window of 5 as well.)
    y_pred = seasonal_average_forecast(data=train, target_dates=target_dates, years_back=years_back_use,day_window=day_window_use)

    # We take the true values.
    y_true = test["count"].values
    
    # Compute the metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append the results of the metrics to the table as well as the fold number.
    results.append({"fold": i, "rmse": rmse, "mape": mape})

# Convert the data to a table for readability.
baseline_results_df = pd.DataFrame(results)

# We also include a new row which consists of the average RMSE and MAPE over each fold.
baseline_results_df.loc["mean"] = ["mean", baseline_results_df["rmse"].mean(), baseline_results_df["mape"].mean()]

baseline_results_df

,fold,rmse,mape
0,0,6.110015,5.187832e-01
1,1,4.792049,1.711072e-01
2,2,4.519542,3.200928e-01
3,3,5.820867,4.201764e-01
4,4,5.803395,3.171943e-01
5,5,7.330541,5.227390e-01
6,6,5.947332,3.256959e-01
7,7,6.018142,2.473551e-01
8,8,6.515253,3.696147e-01
9,9,7.840324,5.171844e-01


## Year Ago Rolling 4 Week Average 

In [9]:
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

## Just saving a copy for later
rs_saved = rs.copy()

In [10]:
# Tired of writing np.sqrt or typing a long name.
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

results = []

for fold, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]

    # Calculate the 4-week rolling average for the training data
    train_sorted = train.sort_values('ds') # making sure to sort it by date
    train_sorted['rolling_4w'] = train_sorted['y'].rolling(window=4, min_periods=1).mean()

    # This part of the code makes the predictions. We use the 'rolling_4w' column of the training set.
    y_pred = []
    y_true = test['y'].values

    for idx, row in test.iterrows():
        # Predict using the latest rolling average from the train data
        prediction = train_sorted['rolling_4w'].iloc[-1]  # Last value in the train rolling avg
        y_pred.append(prediction)
        
    # Calculate RMSE and MAPE for this fold
    fold_rmse = rmse(y_true, y_pred)
    fold_mape = mean_absolute_percentage_error(y_true, y_pred)
    
    results.append({'fold': fold, 'rmse': fold_rmse, 'mape': fold_mape})

rolling4w_results_df = pd.DataFrame(results)

# Optional: add a row for the overall average RMSE and MAPE
overall_rmse = rolling4w_results_df['rmse'].mean()
overall_mape = rolling4w_results_df['mape'].mean()
rolling4w_results_df.loc['mean'] = ['mean', overall_rmse, overall_mape]

In [11]:
rolling4w_results_df

,fold,rmse,mape
0,0,5.355238,4.252300e-01
1,1,8.373001,3.625078e-01
2,2,3.894823,2.717289e-01
3,3,5.993300,3.107589e-01
4,4,6.008922,2.453872e-01
5,5,7.652614,5.443236e-01
6,6,8.599730,3.345505e-01
7,7,8.957738,2.841237e-01
8,8,6.800735,3.947767e-01
9,9,5.257647,2.856080e-01


## Prophet Model

In [12]:
# Create a date range covering 2020 through end of 2025
date_range = pd.date_range(start="2020-01-01", end="2026-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

# Build the DataFrame in the same structure as your original
federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1,
})

holidays = federal_holidays

In [13]:
# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

# Aggregate data to weekly average
rs_weekly = rs.set_index('ds').resample('W')['y'].mean().reset_index()
rs_weekly.columns = ['ds', 'y']

# Create time series split for weekly data (2 week test set = ~28 days equivalent)
tscv_weekly = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=13, test_size=2)

# Disable all logging
logging.disable(logging.CRITICAL)

results = []

for i, (train_index, test_index) in enumerate(tscv_weekly.split(rs_weekly)):
    train = rs_weekly.iloc[train_index]
    test = rs_weekly.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')
    model.fit(train)
    
    # Calculate number of weeks to forecast
    num_forecast_weeks = len(test)
    future = model.make_future_dataframe(periods=num_forecast_weeks, freq='W')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-num_forecast_weeks:].values
    y_true = test['y'].values
    
    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Calculate MAPE
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

# Re-enable logging after Prophet modeling
logging.disable(logging.NOTSET)

# Convert results to a datafrane
prophet_results_df = pd.DataFrame(results)

In [14]:
prophet_results_df.loc['mean'] = ['mean',  prophet_results_df['rmse'].mean(), prophet_results_df['mape'].mean()]

In [15]:
prophet_results_df

,fold,rmse,mape
0,0,3.203010,0.134752
1,1,2.177534,0.119340
2,2,2.656622,0.186958
3,3,2.101044,0.169547
4,4,3.695797,0.379367
5,5,1.651678,0.169892
6,6,1.224583,0.142692
7,7,0.566768,0.074609
8,8,1.512721,0.209642
9,9,1.643472,0.158674


## SARIMA Model with auto_arima parameters.

In [17]:
from pmdarima import auto_arima

In [18]:
def fourier_terms(df, period, n_terms):
    t = np.arange(1, len(df) + 1)
    fourier_df = pd.DataFrame()
    
    for i in range(1, n_terms + 1):
        fourier_df[f'sin_{i}'] = np.sin(2 * np.pi * i * t / period)
        fourier_df[f'cos_{i}'] = np.cos(2 * np.pi * i * t / period)
    
    return fourier_df

In [19]:
# Number of Fourier terms and period (365 for yearly seasonality)
n_terms = 5  # Number of terms for Fourier Terms
period = 365 

In [20]:
# Generate Fourier terms and putting it into a list
fourier_train = fourier_terms(rs, period, n_terms)

# Name it exog since it will serve as exogenous features for SARIMAX. This is the X.
exog = fourier_train

In [21]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Make sure the columns for SARIMA model are renamed.
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

results = []

# Loop through each fold
for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]

    exog_train = exog.iloc[train_index]
    exog_test = exog.iloc[test_index]

    orders = (2,1,3)
    seasonal_orders = (0,0,0,0)

    # model_auto = auto_arima(train['y'], 
    #                         exog=exog_train,  # Exogenous Fourier terms for training data
    #                         seasonal=True, 
    #                         # m=7,  # we choose 7 since we can at least model weekly seasonality. yearly seasonality would not be able to run.
    #                         trace=True, 
    #                         stepwise=True,  # stepwise search to speed up
    #                         suppress_warnings=True, 
    #                         # n_jobs=-1,  # use all available cores for parallel processing
    #                         maxiter=100,  # limit the number of iterations (really important for run time)
    #                         max_p=3, 
    #                         max_q=3, 
    #                         max_P=2, 
    #                         max_Q=2, 
    #                         max_d=2, 
    #                         max_D=1)
    # orders = model_auto.order  # (p, d, q)
    # seasonal_orders = model_auto.seasonal_order  # (P, D, Q, s)
    
    # Fit the SARIMAX model with the exogenous features (Fourier terms)
    model_sarimax = SARIMAX(train['y'], 
                            order=orders,  
                            seasonal_order=seasonal_orders,  
                            exog=exog_train,  # Exogenous Fourier terms for training data
                            enforce_stationarity=False,
                            enforce_invertibility=False)
    
    model_fit = model_sarimax.fit(disp=False)
    
    # Predict for the test period. Have to remember to subtract 1 to get the correct index.
    y_pred = model_fit.predict(start=len(train), end=len(train)+len(test)-1, exog=exog_test, dynamic=False)
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

sarima_results_df = pd.DataFrame(results)

c:\Users\daoke\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\daoke\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\daoke\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\daoke\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\daoke\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximu

In [22]:
sarima_results_df.loc['mean'] = ['mean',  sarima_results_df['rmse'].mean(), sarima_results_df['mape'].mean()]

In [23]:
sarima_results_df

,fold,rmse,mape
0,0,5.006137,3.369887e-01
1,1,6.167585,2.451598e-01
2,2,5.523457,4.001045e-01
3,3,4.760502,3.415601e-01
4,4,5.150684,2.451487e-01
5,5,6.763405,4.694309e-01
6,6,5.279903,2.348319e-01
7,7,7.006311,2.224397e-01
8,8,7.002514,4.056354e-01
9,9,5.873042,3.866227e-01


## Holt-Winters Model

In [24]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

In [25]:
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []
for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    # First we fit the Holt-Winters Exponential Smoothing Model to the training data
    holt_winters = ExponentialSmoothing(train['y'], seasonal='add', seasonal_periods=365).fit(optimized=True)
    
    y_pred = holt_winters.forecast(len(test))
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

hw_results_df = pd.DataFrame(results)

c:\Users\daoke\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\Users\daoke\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\Users\daoke\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\Users\daoke\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\Users\daoke\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\Users\daoke\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:903: ConvergenceWarning: Optimization failed to co

In [26]:
hw_results_df.loc['mean'] = ['mean',  hw_results_df['rmse'].mean(), hw_results_df['mape'].mean()]

In [27]:
hw_results_df

,fold,rmse,mape
0,0,5.728875,4.496844e-01
1,1,7.632696,3.117245e-01
2,2,6.052016,4.158458e-01
3,3,6.744979,4.887953e-01
4,4,7.228464,3.555975e-01
5,5,6.923041,4.293664e-01
6,6,6.506441,2.740266e-01
7,7,8.699921,3.076205e-01
8,8,5.851400,2.969004e-01
9,9,8.230484,5.334110e-01


## XGBoost Model

The XGBoost model requires a bit more preparatory work. Our current dataframe rs is quite bare. We will need to add features for use.

In [28]:
import xgboost as xgb
from xgboost import plot_importance

### Adding Features to XGBoost

In [29]:
def create_features(df):
    # create time series features based on time series index.
    df = df.copy()
    df['dayofweek'] = df.index.dayofweek
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['dayofyear'] = df.index.dayofyear
    df['dayofmonth'] = df.index.day
    df['weekofyear'] = df.index.isocalendar().week
    return df

def add_cyclic(df):
    # features to handly cyclic behavior
    target_map = df['y'].to_dict()
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek']/7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek']/7)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
    return df

def add_lags(df):
    # lags
    target_map = df['y'].to_dict()
    df['lag1'] = (df.index - pd.Timedelta('1 days')).map(target_map)
    df['lag2'] = (df.index - pd.Timedelta('2 days')).map(target_map)
    df['lag3'] = (df.index - pd.Timedelta('3 days')).map(target_map)
    df['lag4'] = (df.index - pd.Timedelta('4 days')).map(target_map)
    df['lag5'] = (df.index - pd.Timedelta('5 days')).map(target_map)
    df['lag6'] = (df.index - pd.Timedelta('6 days')).map(target_map)
    df['lag7'] = (df.index - pd.Timedelta('7 days')).map(target_map)
    df['lag8'] = (df.index - pd.Timedelta('8 days')).map(target_map)
    df['lag9'] = (df.index - pd.Timedelta('9 days')).map(target_map)
    df['lag10'] = (df.index - pd.Timedelta('10 days')).map(target_map)
    df['lag11'] = (df.index - pd.Timedelta('11 days')).map(target_map)
    df['lag12'] = (df.index - pd.Timedelta('12 days')).map(target_map)
    df['lag13'] = (df.index - pd.Timedelta('13 days')).map(target_map)
    df['lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['lag15'] = (df.index - pd.Timedelta('10 days')).map(target_map)
    df['lag16'] = (df.index - pd.Timedelta('11 days')).map(target_map)
    df['lag17'] = (df.index - pd.Timedelta('12 days')).map(target_map)
    df['lag18'] = (df.index - pd.Timedelta('13 days')).map(target_map)
    df['lag19'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['lag20'] = (df.index - pd.Timedelta('10 days')).map(target_map)
    df['lag21'] = (df.index - pd.Timedelta('11 days')).map(target_map)
    df['lag22'] = (df.index - pd.Timedelta('12 days')).map(target_map)
    df['lag23'] = (df.index - pd.Timedelta('13 days')).map(target_map)
    df['lag24'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    return df

def add_seasonal_lags(df):
    # lags of various lengths for different levels of seasonality
    # these are lags on the target itself
    target_map = df['y'].to_dict()
    df['lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)

    df['lag362'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag363'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag364'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag366'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag367'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    
    df['lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)
    df['lag1095'] = (df.index - pd.Timedelta('1095 days')).map(target_map)
    df['lag1460'] = (df.index - pd.Timedelta('1460 days')).map(target_map)
    df['lag1825'] = (df.index - pd.Timedelta('1825 days')).map(target_map)
    return df

def add_moving_averages(df, forecast_horizon_days=14):
    df = df.copy()
    df = df.sort_index()

    # Moving averages (using previous values only)
    df['ma7'] = df['y'].shift(forecast_horizon_days).rolling(window=7).mean()
    df['ma30'] = df['y'].shift(forecast_horizon_days).rolling(window=30).mean()
    df['ma60'] = df['y'].shift(forecast_horizon_days).rolling(window=60).mean()
    df['ma90'] = df['y'].shift(forecast_horizon_days).rolling(window=90).mean()
    df['ma120'] = df['y'].shift(forecast_horizon_days).rolling(window=120).mean()
    df['ma150'] = df['y'].shift(forecast_horizon_days).rolling(window=150).mean()
    df['ma180'] = df['y'].shift(forecast_horizon_days).rolling(window=180).mean()
    df['ma365'] = df['y'].shift(forecast_horizon_days).rolling(window=365).mean()
    
    return df

def add_moving_averages_shifted(df, forecast_horizon_days=14):
    """
    Adds moving averages shifted so that only data before the forecast horizon is used.
    
    Parameters:
    - df: DataFrame with 'y' and datetime index
    - forecast_horizon_days: int, how many days ahead we're forecasting (14 weeks = 98)
    """
    df = df.copy()
    df = df.sort_index()
    
    # shift by forecast horizon + 1 so we never peek into future
    shift_amount = forecast_horizon_days
    
    df['ma7'] = df['y'].shift(shift_amount).rolling(window=7).mean()
    df['ma30'] = df['y'].shift(shift_amount).rolling(window=30).mean()
    df['ma60'] = df['y'].shift(shift_amount).rolling(window=60).mean()
    df['ma90'] = df['y'].shift(shift_amount).rolling(window=90).mean()
    df['ma120'] = df['y'].shift(shift_amount).rolling(window=120).mean()
    df['ma150'] = df['y'].shift(shift_amount).rolling(window=150).mean()
    df['ma180'] = df['y'].shift(shift_amount).rolling(window=180).mean()
    df['ma365'] = df['y'].shift(shift_amount).rolling(window=365).mean()
    
    return df

def add_lagged_differences_shifted(df, forecast_horizon_days=14, lag_list=None):
    """
    Add lagged differences with shift to prevent using future information in cross validation.
    Detects trends and changes in trend without peeking into the future. Our data exhibits changes in trend.

    - forecast_horizon_days: int, how many days ahead we're forecasting (2 weeks = 14)
    - lag_list: list of int, lags in days to compute differences and percent changes
    """
    df = df.copy()
    
    if lag_list is None:
        lag_list = [1, 7, 30, 60, 90, 365]  # default lags

    # Shift all computations by the forecast horizon
    shift_amount = forecast_horizon_days
    
    # Simple differences and percent changes
    for lag in lag_list:
        df[f'y_diff_{lag}'] = df['y'].shift(shift_amount) - df['y'].shift(shift_amount + lag)
        df[f'y_pct_change_{lag}'] = (df['y'].shift(shift_amount) - df['y'].shift(shift_amount + lag)) / df['y'].shift(shift_amount + lag)

    # Differences between successive lag differences to detect trend acceleration
    for i in range(1, len(lag_list)):
        lag_prev = lag_list[i-1]
        lag_curr = lag_list[i]
        df[f'y_diff_lag{lag_prev}_{lag_curr}'] = df[f'y_diff_{lag_curr}'] - df[f'y_diff_{lag_prev}']
    
    return df

In the next two code block, we add weather data to the data set. This is not optimized i.e. we just obtain the weather data in Manhattan and hope that it is representative of the average weather over the whole city.

In [30]:
## Add weather data.

import requests
import pandas as pd

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [31]:
def add_weather_data(df, wd):
    df = df.copy()
    wd = wd.copy()
    
    # Ensure datetime index 
    df.index = pd.to_datetime(df.index)
    wd.index = pd.to_datetime(wd.index)
    
    # Drop unnecessary columns
    if "time" in wd.columns:
        wd = wd.drop(columns=["time"])
    
    # Join on date index
    df = df.join(wd, how="left")
    
    return df

In [32]:
from pandas.tseries.holiday import USFederalHolidayCalendar

def add_federal_holidays(df, custom_holidays=None):
    df = df.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df.index.min(), end=df.index.max())
    
    if custom_holidays:
        for d in custom_holidays:
            if len(d) == 5:  # MM-DD format handling
                years = df.index.year.unique()
                for y in years:
                    holidays = holidays.append(pd.to_datetime([f"{y}-{d}"]))
            else:  # YYYY-MM-DD format handling
                holidays = holidays.append(pd.to_datetime([d]))
    
    holidays = holidays.drop_duplicates().sort_values()
    
    df["is_federal_holiday"] = df.index.isin(holidays).astype(int)
    
    return df

In [33]:
def add_law_flag(df, law_name: str, start_date: str):
    # Adds a binary column to indicate when a new law is active.
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    start_dt = pd.to_datetime(start_date)
    # Create binary column: 1 if date >= start_date, else 0
    df[law_name] = (df.index >= start_dt).astype(int)
    
    return df

In [34]:
# This must be run after importing weather data

def add_more_weather_feature(df):
    target_map = df['apparent_temperature_min'].to_dict()
    df['apparent_temperature_min_lag1'] = (df.index - pd.Timedelta('1 days')).map(target_map)
    df['apparent_temperature_min_lag7'] = (df.index - pd.Timedelta('7 days')).map(target_map)
    df['apparent_temperature_min_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['apparent_temperature_min_lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df['apparent_temperature_min_lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df['apparent_temperature_min_lag17'] = (df.index - pd.Timedelta('17 days')).map(target_map)
    df['apparent_temperature_min_lag18'] = (df.index - pd.Timedelta('18 days')).map(target_map)
    df['apparent_temperature_min_lag19'] = (df.index - pd.Timedelta('19 days')).map(target_map)
    df['apparent_temperature_min_lag20'] = (df.index - pd.Timedelta('20 days')).map(target_map)
    df['apparent_temperature_min_lag21'] = (df.index - pd.Timedelta('21 days')).map(target_map)

    df['apparent_temperature_min_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['apparent_temperature_min_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['apparent_temperature_min_lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['apparent_temperature_min_lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['apparent_temperature_min_lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['apparent_temperature_min_lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)
    df['apparent_temperature_min_lag210'] = (df.index - pd.Timedelta('210 days')).map(target_map)
    df['apparent_temperature_min_lag240'] = (df.index - pd.Timedelta('240 days')).map(target_map)
    df['apparent_temperature_min_lag270'] = (df.index - pd.Timedelta('270 days')).map(target_map)
    df['apparent_temperature_min_lag300'] = (df.index - pd.Timedelta('300 days')).map(target_map)
    df['apparent_temperature_min_lag330'] = (df.index - pd.Timedelta('330 days')).map(target_map)
    df['apparent_temperature_min_lag360'] = (df.index - pd.Timedelta('360 days')).map(target_map)
    df['apparent_temperature_min_lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['apparent_temperature_min_lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)

    target_map = df['temperature_2m_max'].to_dict()
    df['temperature_2m_max_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['temperature_2m_max_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['temperature_2m_max_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)

    return df

In [35]:
rs = rs.set_index('ds')
rs.index = pd.to_datetime(rs.index)

In [36]:
rs = create_features(rs)
rs = add_cyclic(rs)
rs = add_lags(rs)
rs = add_seasonal_lags(rs)
rs = add_moving_averages(rs)
rs = add_weather_data(rs,wd)
rs = add_more_weather_feature(rs)
rs = add_federal_holidays(rs, custom_holidays = ['12-31'])
rs = add_law_flag(rs, law_name='Trash_Law', start_date = '2024-03-01')
rs = add_law_flag(rs, law_name = 'New_Trash_Law', start_date = '2024-11-01')
rs = add_law_flag(rs, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
rs = add_law_flag(rs, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

In [37]:
rs = add_lagged_differences_shifted(rs)

### Features for XGBoost

In [38]:
FEATURES = ['apparent_temperature_min_lag30',
            'apparent_temperature_min_lag60',
            'temperature_2m_max_lag30',
            'temperature_2m_max_lag60',
            'lag15', 'lag16', 'lag17', 'lag18', 'lag19', 'lag30', 'lag60',
            'dayofweek', 'month', 'dayofyear', 'dayofmonth', 'weekofyear',
            'y_pct_change_60', 'y_diff_90', 'y_pct_change_90', 'y_diff_365',
            'y_pct_change_365', 'y_diff_lag1_7', 'y_diff_lag7_30',
            'y_diff_lag30_60', 'y_diff_lag60_90', 'y_diff_lag90_365', 'is_federal_holiday', 'Trash_Law'
            ]

#must be very careful here -- we are trying to forecast 14 days out so

rs.columns

Index(['y', 'dayofweek', 'quarter', 'month', 'year', 'dayofyear', 'dayofmonth',
       'weekofyear', 'dayofweek_sin', 'dayofweek_cos',
       ...
       'y_pct_change_60', 'y_diff_90', 'y_pct_change_90', 'y_diff_365',
       'y_pct_change_365', 'y_diff_lag1_7', 'y_diff_lag7_30',
       'y_diff_lag30_60', 'y_diff_lag60_90', 'y_diff_lag90_365'],
      dtype='object', length=117)

### Add default parameters for XGBoost

In [39]:
params = {'objective': 'reg:squarederror',
         'eval_metric': 'rmse',
         'booster': 'gbtree',
        # 'base_score': 0.5, 
         'n_estimators': 500, 
        # 'min_child_weight': 7, 
         'learning_rate': 0.1,
        #  'max_depth': 8, 
        #  'subsample': 1,
        #  'colsample_bytree': 0.96,
        #  'colsample_bylevel': 0.6, 
        #  'colsample_bynode': 0.9, 
        #  'reg_alpha': 2.2, 
        #  'gamma': 100, 
        #  'reg_lambda': 0.18,
        #  'early_stopping_rounds': 100, 
        }

## uncomment to use GPU
params.update({
    "tree_method": "hist",
    "device": "cuda"
})


### Results for XGBoost Model

In [40]:
print(FEATURES)
print(params)
TARGET = 'y'

# Gotta make sure the features and parameters exist.

reg = xgb.XGBRegressor(**params)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    reg.fit(train[FEATURES], train[TARGET])
    y_pred = reg.predict(test[FEATURES])
    y_true = test[TARGET].values
    
    # Our metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})
    print(results)

xgb_results_df = pd.DataFrame(results)
mean_rmse = xgb_results_df['rmse'].mean()
mean_mape = xgb_results_df['mape'].mean()
xgb_results_df.loc['mean'] = ['mean', mean_rmse, mean_mape]

['apparent_temperature_min_lag30', 'apparent_temperature_min_lag60', 'temperature_2m_max_lag30', 'temperature_2m_max_lag60', 'lag15', 'lag16', 'lag17', 'lag18', 'lag19', 'lag30', 'lag60', 'dayofweek', 'month', 'dayofyear', 'dayofmonth', 'weekofyear', 'y_pct_change_60', 'y_diff_90', 'y_pct_change_90', 'y_diff_365', 'y_pct_change_365', 'y_diff_lag1_7', 'y_diff_lag7_30', 'y_diff_lag30_60', 'y_diff_lag60_90', 'y_diff_lag90_365', 'is_federal_holiday', 'Trash_Law']
{'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'booster': 'gbtree', 'n_estimators': 500, 'learning_rate': 0.1, 'tree_method': 'hist', 'device': 'cuda'}
[{'fold': 0, 'rmse': 4.61088304581218, 'mape': 0.3859383165836334}]
[{'fold': 0, 'rmse': 4.61088304581218, 'mape': 0.3859383165836334}, {'fold': 1, 'rmse': 7.491704294734683, 'mape': 0.31083551049232483}]
[{'fold': 0, 'rmse': 4.61088304581218, 'mape': 0.3859383165836334}, {'fold': 1, 'rmse': 7.491704294734683, 'mape': 0.31083551049232483}, {'fold': 2, 'rmse': 4.6316921172

XGBoostError: [08:41:09] C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\data\data.cc:1120: Check failed: valid: Input data contains `inf` or a value too large, while `missing` is not set to `inf`

In [ ]:
xgb_results_df

In [ ]:
model = reg.fit(train[FEATURES], train[TARGET])
importance_types = ['weight', 'gain', 'cover', 'total_gain', 'total_cover']
for importance_type in importance_types:
    fig, ax = plt.subplots(figsize=(10, 6))
    xgb.plot_importance(model, ax=ax, importance_type=importance_type, grid=False, show_values=False)
    plt.xlabel(f'{importance_type}'.capitalize())
    plt.title(f'Feature Importance ({importance_type})')
    plt.show()

## XGBoosted Prophet Model

In [41]:
# Recall the copy that was saved.
rs_saved

,ds,y
0,2020-01-01,4
1,2020-01-02,7
2,2020-01-03,16
3,2020-01-04,10
4,2020-01-05,5
...,...,...
2246,2026-02-24,7
2247,2026-02-25,8
2248,2026-02-26,9
2249,2026-02-27,17


In [42]:
rs = rs_saved

In [43]:
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    # Split the dataset into training and testing sets
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    # Fit Prophet on the training data
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')
    model.fit(train)
    
    # Make predictions on the training set to calculate residuals
    train_future = model.make_future_dataframe(periods=0, freq='D')  # Use periods=0 to only use the training data
    train_forecast = model.predict(train_future)
    
    # Calculate residuals (actual - predicted) on the training data
    train_residuals = train['y'].values - train_forecast['yhat'][:len(train)].values
    
    # Build a new DataFrame of residuals
    residuals_df = pd.DataFrame({'ds': train['ds'], 'y': train_residuals })

    train = train.set_index('ds')
    train.index = pd.to_datetime(train.index)
    train = create_features(train)
    train = add_cyclic(train)
    train = add_lags(train)
    train = add_seasonal_lags(train)
    train = add_moving_averages(train)
    train = add_weather_data(train,wd)
    train = add_more_weather_feature(train)
    train = add_federal_holidays(train, custom_holidays = ['12-31'])
    train = add_law_flag(train, law_name='Trash_Law', start_date = '2024-03-01')
    train = add_law_flag(train, law_name = 'New_Trash_Law', start_date = '2024-11-01')
    train = add_law_flag(train, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
    train = add_law_flag(train, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

    X_train_residuals = train[FEATURES]
    y_train_residuals = residuals_df['y']
    
    xgb_model = xgb.XGBRegressor(**params)
    xgb_model.fit(X_train_residuals, y_train_residuals)
    

    test = test.set_index('ds')
    test.index = pd.to_datetime(test.index)
    test = create_features(test)
    test = add_cyclic(test)
    test = add_lags(test)
    test = add_seasonal_lags(test)
    test = add_moving_averages(test)
    test = add_weather_data(test,wd)
    test = add_more_weather_feature(test)
    test = add_federal_holidays(test, custom_holidays = ['12-31'])
    test = add_law_flag(test, law_name='Trash_Law', start_date = '2024-03-01')
    test = add_law_flag(test, law_name = 'New_Trash_Law', start_date = '2024-11-01')
    test = add_law_flag(test, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
    test = add_law_flag(test, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

    # Predict residuals using XGBoost for the test set
    X_test = test[FEATURES]  # Features for the test set
    xgb_residual_preds = xgb_model.predict(X_test)
    
    # Forecast using Prophet on the test set
    future = model.make_future_dataframe(periods=len(test), freq='D')
    prophet_forecast = model.predict(future)
    
    # Combine Prophet's forecast and XGBoost's residual prediction
    y_pred = prophet_forecast['yhat'][-len(test):].values + xgb_residual_preds
    
    # Actual values for the test set
    y_true = test['y'].values
    
    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Calculate MAPE
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Store the results for this fold
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

    ## Uncomment code below if you want to have plots on feature importance.
    # fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(10, 30))
    # plot_importance(xgb_model, ax=ax1, importance_type='gain')
    # ax1.set_title('Gain-based Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax2, importance_type='weight')
    # ax2.set_title('Split-based Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax3, importance_type='cover')
    # ax3.set_title('Cover Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax4, importance_type='total_gain')
    # ax4.set_title('Total Gain Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax5, importance_type='total_cover')
    # ax5.set_title('Total Cover Importance', fontsize=12)

    plt.show()

# Convert the results into a DataFrame
prophet_xgb_results_df = pd.DataFrame(results)

prophet_xgb_results_df = pd.DataFrame(results)
mean_rmse = prophet_xgb_results_df['rmse'].mean()
mean_mape = prophet_xgb_results_df['mape'].mean()
prophet_xgb_results_df.loc['mean'] = ['mean', mean_rmse, mean_mape]

08:42:31 - cmdstanpy - INFO - Chain [1] start processing
08:42:31 - cmdstanpy - INFO - Chain [1] done processing


KeyError: "['y_pct_change_60', 'y_diff_90', 'y_pct_change_90', 'y_diff_365', 'y_pct_change_365', 'y_diff_lag1_7', 'y_diff_lag7_30', 'y_diff_lag30_60', 'y_diff_lag60_90', 'y_diff_lag90_365'] not in index"

In [ ]:
prophet_xgb_results_df

# Conclusions on Model Comparisons

## Results Table

In [44]:
# We make a dictionary of models and their results to make it easier to iterate over.
models = {
    'baseline': baseline_results_df,
    'rolling4w': rolling4w_results_df,
    'prophet': prophet_results_df,
    'sarima': sarima_results_df,
    'hw': hw_results_df,
    #'xgb': xgb_results_df,
    #'prophet+xgb': prophet_xgb_results_df
}

all_results = []
for model_name, df in models.items():
    df['model'] = model_name
    all_results.append(df)

# Put all of the dataframes together into one dataframe for display
final_results_df = pd.concat(all_results, ignore_index=True)

# Make a pivot table so that we display rmse, mape and then each of the models and their results.
final_table = final_results_df.pivot(index='fold', columns='model', values=['rmse', 'mape'])
final_table.index = final_table.index.where(final_table.index != '-', 'mean')

final_table

rmse                                                  mape  \
model   baseline        hw   prophet rolling4w    sarima      baseline   
fold                                                                     
0       6.110015  5.728875  3.203010  5.355238  5.006137  5.187832e-01   
1       4.792049  7.632696  2.177534  8.373001  6.167585  1.711072e-01   
2       4.519542  6.052016  2.656622  3.894823  5.523457  3.200928e-01   
3       5.820867  6.744979  2.101044  5.993300  4.760502  4.201764e-01   
4       5.803395  7.228464  3.695797  6.008922  5.150684  3.171943e-01   
5       7.330541  6.923041  1.651678  7.652614  6.763405  5.227390e-01   
6       5.947332  6.506441  1.224583  8.599730  5.279903  3.256959e-01   
7       6.018142  8.699921  0.566768  8.957738  7.006311  2.473551e-01   
8       6.515253  5.851400  1.512721  6.800735  7.002514  3.696147e-01   
9       7.840324  8.230484  1.643472  5.257647  5.873042  5.171844e-01   
10      9.310988  7.666122  1.962190  5.630751  5.962313  6.290299e-01   
11      8.895986  6.026421  1.854723  5.490251  5.476072  6.089048e-01   
12      8.002407  8.319851  2.005822  8.568526  7.821556  4.195949e-01   
13      6.797736  5.970185       NaN  6.602894  5.940978  4.752848e-01   
14      7.611748  5.825099       NaN  5.500000  5.677783  5.331502e-01   
15      8.421691  5.931978       NaN  6.366065  4.338090  8.997112e-01   
16      7.654961  4.239907       NaN  4.689464  4.133086  7.062848e-01   
17     10.062572  6.139507       NaN  7.188781  4.694671  1.681684e+00   
18      7.677977  4.450710       NaN  3.668154  3.972458  9.293678e-01   
19      6.678495  5.035076       NaN  5.628372  3.965338  3.824485e+15   
20      6.710397  3.217965       NaN  2.950484  2.534887  1.169345e+00   
21      6.719962  5.363270       NaN  4.156535  3.667662  1.850322e+00   
22      5.608994  4.529405       NaN  5.229552  4.456958  5.981685e-01   
23      7.456576  4.045715       NaN  3.935870  3.864979  1.286946e+00   
24      8.015235  2.764322       NaN  2.041970  4.066910  1.393778e+00   
25      6.383076  3.849932       NaN  5.143477  4.002202  8.871942e-01   
mean    7.027164  5.883607  2.019690  5.757111  5.119596  1.470956e+14   

                                                           
model            hw   prophet     rolling4w        sarima  
fold                                                       
0      4.496844e-01  0.134752  4.252300e-01  3.369887e-01  
1      3.117245e-01  0.119340  3.625078e-01  2.451598e-01  
2      4.158458e-01  0.186958  2.717289e-01  4.001045e-01  
3      4.887953e-01  0.169547  3.107589e-01  3.415601e-01  
4      3.555975e-01  0.379367  2.453872e-01  2.451487e-01  
5      4.293664e-01  0.169892  5.443236e-01  4.694309e-01  
6      2.740266e-01  0.142692  3.345505e-01  2.348319e-01  
7      3.076205e-01  0.074609  2.841237e-01  2.224397e-01  
8      2.969004e-01  0.209642  3.947767e-01  4.056354e-01  
9      5.334110e-01  0.158674  2.856080e-01  3.866227e-01  
10     4.326266e-01  0.234153  2.703794e-01  3.724050e-01  
11     3.665332e-01  0.270296  2.610070e-01  3.685401e-01  
12     2.420034e-01  0.211838  1.992474e-01  1.959620e-01  
13     3.784072e-01       NaN  3.529541e-01  3.585091e-01  
14     3.518706e-01       NaN  3.884974e-01  3.649979e-01  
15     6.165255e-01       NaN  6.639706e-01  4.481878e-01  
16     3.409124e-01       NaN  4.302836e-01  3.583617e-01  
17     9.310257e-01       NaN  1.202054e+00  6.544343e-01  
18     3.973464e-01       NaN  4.107391e-01  3.660545e-01  
19     3.052524e+15       NaN  3.699385e+15  2.720905e+15  
20     4.191831e-01       NaN  4.992334e-01  4.090699e-01  
21     1.061226e+00       NaN  1.274620e+00  1.034690e+00  
22     4.117504e-01       NaN  3.352837e-01  2.774093e-01  
23     5.963330e-01       NaN  6.316356e-01  6.199490e-01  
24     4.003346e-01       NaN  2.220805e-01  6.849951e-01  
25     3.583801e-01       NaN  4.251677e-01  3.805269e-01  
mean   1.174048e+14  0.189366  1.422841e+14  1.0465

## Summary

In the above table, we see that the Prophet model had the best average RMSE, but the margins are quite slim so far. Prophet beat out Sarima by at best 0.2 on average.

# Neural Prophet

In [ ]:
pip install neuralprophet

In [ ]:
from neuralprophet import NeuralProphet

import numpy as np
np.NaN = np.nan


# the following packages are meant to turn off a bunch of the warnings and ERRORs that pop up while running NeuralProphet.
# the errors that do show up are not all that important and a lot is due to outdated packages.
import warnings
import logging

warnings.filterwarnings("ignore")

logging.getLogger("neuralprophet").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("NP").setLevel(logging.ERROR)

In [ ]:
# set up the time series split
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2026-02-28.
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to MANHATTAN

rs = rs[rs['borough']=='MANHATTAN']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

In [ ]:
## Add weather data.

import requests
import pandas as pd

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

def add_weather_data_no_index(df,wd):
    if "time" in wd.columns:
        wd = wd.drop(columns=["time"])

    for column in wd.columns:
        df[column] = wd[column].values

    return df

In [ ]:
regressed_features = ['apparent_temperature_max', 
    #'apparent_temperature_min', 
    #'apparent_temperature_mean', 
    #'snowfall_sum'
    ]
wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(
    wd[['ds'] + regressed_features],
    on="ds",
    how="left"
)

rs

In [ ]:
df = rs.copy()

df["ds"] = pd.to_datetime(df["ds"])

plt.figure(figsize=(15,6))
plt.plot(df["ds"], np.log(df["y"]))

plt.xlabel("Date")
plt.ylabel("Sightings")
plt.title("Sightings over Time")
plt.grid(True)

plt.show()

In [ ]:
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):

    train = rs.iloc[train_index].copy()
    test = rs.iloc[test_index].copy()

    train = train.dropna(subset=["y"])

    model = NeuralProphet(yearly_seasonality=True, 
                          weekly_seasonality=True, 
                          epochs = 40,
                          accelerator = 'auto',
                          n_lags=7)
    model = model.add_country_holidays(country_name="US")

    model.add_lagged_regressor('apparent_temperature_max', n_lags=12)


    # merge regressors correctly
    # train = train.merge(wd[['ds'] + regressed_features], on="ds", how="left")

    model.fit(train, freq="D", progress="off")

    # build dataframe containing future regressors
    future = pd.concat([train[['ds','y'] + regressed_features], test[['ds','y']].merge(wd[['ds'] + regressed_features], on="ds", how="left")])
    forecast = model.predict(future)

    y_pred = forecast["yhat1"].iloc[-len(test):].values
    y_pred = np.round(y_pred)
    y_true = test["y"].values

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)

    results.append({"fold": i, "rmse": rmse, "mape": mape})

neural_prophet_results_df = pd.DataFrame(results)
neural_prophet_results_df.loc["mean"] = ["mean", neural_prophet_results_df["rmse"].mean(), neural_prophet_results_df["mape"].mean()]
neural_prophet_results_df

In [ ]:
model.plot(forecast)

In [ ]:
model.plot_parameters()